[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# Introducción a *Polars*.

## ¿Qué es *Polars*?

[*Polars*](https://pola.rs/) es una biblioteca de *dataframes* escrita en *Rust*. Usa el formato columnar de *Apache Arrow* (estudiado en el *notebook* 34) como representación en memoria y reparte el trabajo entre todos los núcleos del procesador de forma automática.

Sus características principales son:

* **Sin índice de renglones.** Un *dataframe* de *Polars* es sólo un conjunto de columnas; los renglones se identifican por su posición o por el valor de alguna columna.
* **Expresiones.** Las operaciones se describen con expresiones, como `pl.col('precio') * 2`, que se evalúan dentro de un contexto (`select()`, `with_columns()`, `filter()`, `group_by().agg()`). Esto permite a *Polars* optimizar y paralelizar el cálculo.
* **Tipos estrictos.** Cada columna tiene un tipo fijo y los nulos se representan igual para cualquier tipo.
* **Ejecución inmediata o diferida.** Además del modo inmediato (*eager*), que se usa en este *notebook*, *Polars* tiene un modo diferido (*lazy*) que optimiza una consulta completa antes de ejecutarla. Se estudia en el *notebook* 38.

### *Pandas* 3 y *Polars*.

| Aspecto | *Pandas* 3 | *Polars* |
| :--- | :--- | :--- |
| Implementación | *Python*, *C* y *Cython* sobre *NumPy* | *Rust* |
| Formato en memoria | *NumPy*; *Arrow* para texto y, opcionalmente, para otras columnas | *Arrow* |
| Índice de renglones | Sí | No |
| Paralelismo | La mayoría de las operaciones usan un solo hilo | Multihilo automático |
| Modo de ejecución | Inmediato | Inmediato (*eager*) o diferido (*lazy*) |
| Estilo de código | Métodos e indexación con corchetes | Expresiones dentro de contextos |
| Datos más grandes que la memoria | No | En muchos casos, con el motor *streaming* |
| Ecosistema | Muy amplio y maduro | Más reciente; se integra con otras herramientas mediante *Arrow* |

*Polars* suele ser más rápido que *Pandas*, sobre todo con datos grandes y en operaciones de agrupación y combinación de tablas. El *notebook* 38 incluye, como autoestudio, una medición. *Pandas* sigue siendo la opción con más integraciones (por ejemplo, con *scikit-learn*, *statsmodels* o *Seaborn*), y la conversión entre ambas bibliotecas es sencilla.

[Guía de usuario de *Polars*](https://docs.pola.rs/user-guide/) · [Referencia de la *API*](https://docs.pola.rs/api/python/stable/reference/index.html)

* La siguiente celda instala en el entorno actual la versión de *Polars* fijada en `requirements.txt` (2.1.0), en caso de que no esté disponible. Fijar la versión garantiza que los resultados y los mensajes coincidan con los de este *notebook*.

In [ ]:
%pip install polars==2.1.0

* La siguiente celda importa `polars` con el alias convencional `pl` y el módulo de selectores `polars.selectors` con el alias `cs`, que se explica más adelante. `pl.Config.set_fmt_float('full')` hace que los números de punto flotante se muestren completos en lugar de en notación científica.

In [ ]:
import os

import polars as pl
import polars.selectors as cs

pl.Config.set_fmt_float('full')
print(f"Polars {pl.__version__}")

* La siguiente celda define las rutas de los datos de ejemplo, descritos en el *notebook* 34, y crea el directorio `data_tmp` para los archivos que se generen.

In [ ]:
ruta_ventas = os.path.join('data', 'ventas.csv')
ruta_tiendas = os.path.join('data', 'tiendas.csv')
dir_tmp = 'data_tmp'
os.makedirs(dir_tmp, exist_ok=True)

## *Dataframes* y series.

Un *dataframe* de *Polars* (`pl.DataFrame`) es una tabla formada por columnas con nombre; cada columna es una serie (`pl.Series`) de un único tipo.

```
pl.DataFrame(<datos>, schema=<schema>)
```

Donde:

* `<datos>` es un diccionario de listas, una lista de diccionarios, un arreglo de *NumPy* o un *dataframe* de *Pandas*, entre otros.
* `<schema>` es opcional: un diccionario `{nombre: tipo}` con los tipos de las columnas. Si se omite, *Polars* infiere los tipos.

| Atributo o método | Descripción |
| :--- | :--- |
| [`df.shape`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.shape.html) | Tupla con el número de renglones y de columnas. |
| [`df.height`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.height.html) / [`df.width`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.width.html) | Número de renglones / de columnas. |
| [`df.columns`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.columns.html) | Lista con los nombres de las columnas. |
| [`df.schema`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.schema.html) | Nombres y tipos de las columnas. |
| [`df.head(n)`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.head.html) / [`df.tail(n)`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.tail.html) | Primeros / últimos `n` renglones (5 por omisión). |
| [`df.glimpse()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.glimpse.html) | Vista compacta: una línea por columna con su tipo y primeros valores. |
| [`df.describe()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.describe.html) | Estadísticas descriptivas de cada columna. |

**Ejemplos:**

* La siguiente celda crea un *dataframe* pequeño a partir de un diccionario de listas.

In [ ]:
productos = pl.DataFrame({
    'producto': ['Cuaderno de espiral', 'Mochila escolar', 'Balón de fútbol', 'Cafetera de goteo'],
    'categoria': ['Papelería', 'Papelería', 'Deportes', 'Hogar'],
    'precio': [65.0, 649.0, 459.0, 799.0],
    'existencias': [120, 35, 0, 18],
})
productos

* La siguiente celda muestra el *schema*. Los tipos de *Polars* se escriben con mayúscula inicial: `String`, `Float64`, `Int64`.

In [ ]:
productos.schema

* La siguiente celda crea el mismo *dataframe* indicando tipos más pequeños para `precio` y `existencias`.

In [ ]:
pl.DataFrame(
    productos.to_dict(),
    schema={'producto': pl.String, 'categoria': pl.String, 'precio': pl.Float32, 'existencias': pl.Int16},
).schema

* La siguiente celda obtiene la columna `precio` con corchetes. El resultado es una serie, que tiene nombre y tipo pero no índice.

In [ ]:
precios = productos['precio']
precios

* Como no hay índice, los renglones se obtienen por posición. La siguiente celda obtiene el segundo renglón como tupla con `row()` y los dos primeros como *dataframe* con `slice()`.

In [ ]:
print(productos.row(1))
productos.slice(0, 2)

## Lectura y escritura de archivos.

| Formato | Lectura | Escritura |
| :--- | :--- | :--- |
| *CSV* | [`pl.read_csv()`](https://docs.pola.rs/api/python/stable/reference/api/polars.read_csv.html) | [`df.write_csv()`](https://docs.pola.rs/api/python/stable/reference/api/polars.DataFrame.write_csv.html) |
| *Parquet* | [`pl.read_parquet()`](https://docs.pola.rs/api/python/stable/reference/api/polars.read_parquet.html) | [`df.write_parquet()`](https://docs.pola.rs/api/python/stable/reference/api/polars.DataFrame.write_parquet.html) |
| *Feather* / *Arrow IPC* | [`pl.read_ipc()`](https://docs.pola.rs/api/python/stable/reference/api/polars.read_ipc.html) | [`df.write_ipc()`](https://docs.pola.rs/api/python/stable/reference/api/polars.DataFrame.write_ipc.html) |
| *JSON* | [`pl.read_json()`](https://docs.pola.rs/api/python/stable/reference/api/polars.read_json.html) | [`df.write_json()`](https://docs.pola.rs/api/python/stable/reference/api/polars.DataFrame.write_json.html) |
| *Excel* | [`pl.read_excel()`](https://docs.pola.rs/api/python/stable/reference/api/polars.read_excel.html) | [`df.write_excel()`](https://docs.pola.rs/api/python/stable/reference/api/polars.DataFrame.write_excel.html) |
| Base de datos | [`pl.read_database()`](https://docs.pola.rs/api/python/stable/reference/api/polars.read_database.html) | [`df.write_database()`](https://docs.pola.rs/api/python/stable/reference/api/polars.DataFrame.write_database.html) |

Cada función `read_*` tiene una equivalente `scan_*` (por ejemplo, `pl.scan_parquet()`) para el modo diferido, que se estudia en el *notebook* 38.

Algunos argumentos útiles de `pl.read_csv()` son:

* `try_parse_dates=True` convierte a fecha las columnas de texto con formato de fecha.
* `schema_overrides={columna: tipo}` fija el tipo de algunas columnas; las demás se infieren.
* `columns=[...]` lee sólo las columnas indicadas.

**Ejemplos:**

* La siguiente celda lee `data/ventas.csv`, convierte las fechas y fija el tipo de `tienda_id`.

In [ ]:
ventas = pl.read_csv(ruta_ventas, try_parse_dates=True, schema_overrides={'tienda_id': pl.Int16})
ventas.shape

* La siguiente celda muestra una vista compacta del *dataframe* con `glimpse()`.

In [ ]:
ventas.glimpse()

* La siguiente celda calcula estadísticas descriptivas de las columnas numéricas con `describe()`.

In [ ]:
ventas.select('cantidad', 'precio_unitario', 'descuento').describe()

* La siguiente celda escribe el *dataframe* en formato *Parquet* y lo lee de nuevo. A diferencia del *CSV*, el archivo *Parquet* conserva los tipos de las columnas.

In [ ]:
ruta_parquet = os.path.join(dir_tmp, 'ventas_polars.parquet')
ventas.write_parquet(ruta_parquet)

pl.read_parquet(ruta_parquet).schema == ventas.schema

## Contextos y expresiones.

En *Polars*, la mayoría de las operaciones se escriben con **expresiones**. Una expresión describe un cálculo sobre una o más columnas, por ejemplo `pl.col('precio') * 2`, que duplica cada precio, pero no hace nada por sí sola: se evalúa cuando se usa dentro de un **contexto**.

| Contexto | Resultado |
| :--- | :--- |
| [`df.select(<expresiones>)`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.select.html) | Un *dataframe* sólo con las columnas que producen las expresiones. |
| [`df.with_columns(<expresiones>)`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.with_columns.html) | El *dataframe* original con columnas agregadas o reemplazadas. |
| [`df.filter(<expresiones>)`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.filter.html) | Los renglones en los que las expresiones booleanas son verdaderas. |
| [`df.group_by(<claves>).agg(<expresiones>)`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.group_by.html) | Un renglón por grupo, con una columna por expresión. |

Todos los contextos regresan un *dataframe* nuevo; el original no se modifica. Dentro de un contexto, *Polars* puede evaluar varias expresiones en paralelo.

[`pl.col(<nombre>)`](https://docs.pola.rs/api/python/stable/reference/expressions/col.html) es la expresión más básica: hace referencia a una columna. El *notebook* 36 estudia las expresiones a fondo.

* La siguiente celda define la variable `tasa_impuesto` con una tasa de impuesto de ejemplo del 15 % (`0.15`), que se usará para calcular el precio con impuesto. Es un dato de ejemplo: la tasa real de un impuesto sobre las ventas depende del país y del producto, y algunos productos no pagan impuesto.

In [ ]:
tasa_impuesto = 0.15

* La siguiente celda define la expresión `expresion`, que calcula el precio unitario con impuesto, y muestra que una expresión fuera de un contexto es sólo una descripción del cálculo.

In [ ]:
expresion = pl.col('precio_unitario') * (1 + tasa_impuesto)
expresion

* La siguiente celda evalúa la misma expresión dentro de `select()`. Por omisión, la columna resultante conserva el nombre de la primera columna de la expresión; `alias()` le asigna otro nombre.

In [ ]:
ventas.select(
    'producto',
    pl.col('precio_unitario'),
    expresion.round(2).alias('precio_con_impuesto'),
).head(3)

## Selección de columnas.

Dentro de `select()` (y de los demás contextos) las columnas pueden indicarse de varias formas:

| Forma | Selecciona |
| :--- | :--- |
| `'producto'` o `pl.col('producto')` | Una columna por nombre. |
| `pl.col('producto', 'cantidad')` | Varias columnas por nombre. |
| `pl.col('^cli.*$')` | Las columnas cuyo nombre coincide con la expresión regular. **El patrón debe empezar con `^` y terminar con `$`**; de lo contrario se interpreta como un nombre literal. |
| `pl.col(pl.Float64)` | Las columnas de un tipo. |
| `pl.all()` | Todas las columnas. |
| `pl.exclude('venta_id', 'cliente_email')` | Todas, excepto las indicadas. |

### Selectores.

El módulo [`polars.selectors`](https://docs.pola.rs/api/python/stable/reference/selectors.html), que se importa como `cs`, ofrece funciones para seleccionar columnas por tipo o por nombre. Los selectores se pueden combinar con operaciones de conjuntos: `|` (unión), `&` (intersección) y `-` (diferencia).

| Selector | Selecciona |
| :--- | :--- |
| `cs.numeric()` | Columnas numéricas. |
| `cs.string()` | Columnas de texto. |
| `cs.temporal()` | Columnas de fecha y hora. |
| `cs.starts_with('...')` / `cs.ends_with('...')` / `cs.contains('...')` | Columnas cuyo nombre empieza con, termina con o contiene un texto. |
| `cs.by_name(...)` / `cs.by_dtype(...)` | Columnas por nombre o por tipo. |

**Ejemplos:**

* La siguiente celda selecciona dos columnas por nombre.

In [ ]:
ventas.select('producto', 'cantidad').head(3)

* La siguiente celda selecciona con una expresión regular las columnas cuyo nombre termina en `_id`.

In [ ]:
ventas.select(pl.col('^.*_id$')).head(3)

* La siguiente celda intenta usar un patrón sin `^` y `$`. *Polars* lo interpreta como el nombre literal de una columna y desencadena una excepción de tipo `ColumnNotFoundError`.

In [ ]:
ventas.select(pl.col('.*_id'))

* La siguiente celda selecciona todas las columnas excepto `venta_id` y `cliente_email`.

In [ ]:
ventas.select(pl.exclude('venta_id', 'cliente_email')).head(3)

* La siguiente celda usa selectores para obtener las columnas numéricas, excepto los identificadores.

In [ ]:
ventas.select(cs.numeric() - cs.ends_with('_id')).head(3)

* La siguiente celda combina un selector de texto y uno de fecha con `|`.

In [ ]:
ventas.select(cs.string() | cs.temporal()).head(3)

## Filtros.

[`df.filter()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.filter.html) conserva los renglones en los que una expresión booleana es verdadera.

```
df.filter(<condición_1>, <condición_2>, ...)
```

Donde cada `<condición>` es una expresión booleana. Si se indican varias, deben cumplirse todas.

| Operación | Expresión |
| :--- | :--- |
| Comparación | `pl.col('cantidad') > 3`, `pl.col('canal') == 'en línea'` |
| Y, o, negación | `(<cond_1>) & (<cond_2>)`, `(<cond_1>) \| (<cond_2>)`, `~(<cond>)` |
| Pertenencia a una lista | `pl.col('categoria').is_in(['Hogar', 'Deportes'])` |
| Rango (incluye los extremos) | `pl.col('precio_unitario').is_between(100, 500)` |
| Nulos | `pl.col('descuento').is_null()`, `pl.col('descuento').is_not_null()` |

Los paréntesis son necesarios al combinar condiciones con `&`, `|` y `~`, porque estos operadores tienen mayor precedencia que las comparaciones.

**Ejemplos:**

* La siguiente celda obtiene las ventas en línea de más de 3 piezas.

In [ ]:
ventas.filter(pl.col('canal') == 'en línea', pl.col('cantidad') > 3).select(
    'fecha_hora', 'canal', 'producto', 'cantidad'
).head(3)

* La siguiente celda obtiene las ventas de `Hogar` o `Deportes` con precio unitario entre 300 y 500 unidades monetarias.

In [ ]:
ventas.filter(
    pl.col('categoria').is_in(['Hogar', 'Deportes']),
    pl.col('precio_unitario').is_between(300, 500),
).select('producto', 'categoria', 'precio_unitario').head(5)

* La siguiente celda obtiene las ventas de diciembre de 2025 o de más de 4 piezas, usando `|`.

In [ ]:
ventas.filter(
    (pl.col('fecha_hora') >= pl.datetime(2025, 12, 1)) | (pl.col('cantidad') > 4)
).height

## Transformaciones.

| Método | Descripción |
| :--- | :--- |
| [`df.with_columns()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.with_columns.html) | Agrega columnas o reemplaza las existentes. |
| [`df.rename()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.rename.html) | Renombra columnas con un diccionario `{actual: nuevo}`. |
| [`df.drop()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.drop.html) | Elimina columnas. |
| [`df.sort()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.sort.html) | Ordena por una o varias columnas. |
| [`df.unique()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.unique.html) | Elimina renglones duplicados, considerando todas las columnas o sólo las indicadas en `subset`. |
| [`df.top_k()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.top_k.html) | Los `k` renglones con los valores más altos de una columna, sin ordenar todo el *dataframe*. |

### El método `with_columns()`.

```
df.with_columns(
    <expresión_1>.alias('<nombre_1>'),
    <nombre_2>=<expresión_2>,
    ...
)
```

Donde cada columna nueva se nombra con `alias()` o con un argumento por nombre. Si el nombre coincide con una columna existente, ésta se reemplaza. Las expresiones de una misma llamada se calculan en paralelo, por lo que no pueden usar una columna creada en la misma llamada; para eso se encadena otro `with_columns()`.

**Ejemplos:**

* La siguiente celda agrega el importe bruto de cada venta y el importe neto, que descuenta el porcentaje de descuento. Como `descuento` tiene nulos, éstos se tratan como `0` con `fill_null()`. El importe neto usa el importe bruto, por lo que se calcula en un segundo `with_columns()`.

In [ ]:
ventas = (
    ventas
    .with_columns(importe=pl.col('cantidad') * pl.col('precio_unitario'))
    .with_columns(
        (pl.col('importe') * (1 - pl.col('descuento').fill_null(0))).round(2).alias('importe_neto')
    )
)
ventas.select('cantidad', 'precio_unitario', 'descuento', 'importe', 'importe_neto').head(5)

* La siguiente celda renombra dos columnas y elimina otra. El *dataframe* `ventas` no cambia porque el resultado no se asigna.

In [ ]:
ventas.rename({'precio_unitario': 'precio', 'cliente_email': 'cliente'}).drop('venta_id').columns

* La siguiente celda ordena por categoría (ascendente) e importe neto (descendente).

In [ ]:
ventas.sort(['categoria', 'importe_neto'], descending=[False, True]).select(
    'categoria', 'producto', 'importe_neto'
).head(5)

* La siguiente celda obtiene las tres ventas con mayor importe neto con `top_k()`.

In [ ]:
ventas.top_k(3, by='importe_neto').select('fecha_hora', 'producto', 'cantidad', 'importe_neto')

* La siguiente celda obtiene las combinaciones únicas de categoría y producto, ordenadas.

In [ ]:
ventas.select('categoria', 'producto').unique().sort('categoria', 'producto')

## Valores nulos.

*Polars* representa los valores faltantes como `null` en cualquier tipo de dato. `NaN` (*not a number*) es distinto: es un valor de punto flotante válido que resulta de operaciones como `0 / 0`, y no se considera nulo.

| Método o expresión | Descripción |
| :--- | :--- |
| [`df.null_count()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.null_count.html) | Número de nulos por columna. |
| `pl.col(...).is_null()` / `is_not_null()` | Expresión booleana para filtrar. |
| [`pl.col(...).fill_null(<valor>)`](https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.fill_null.html) | Reemplaza los nulos por un valor o con una estrategia: `strategy='forward'`, `'backward'`, `'mean'`, `'min'`, `'max'` o `'zero'`. |
| [`df.drop_nulls(subset=...)`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.drop_nulls.html) | Elimina los renglones con nulos en las columnas indicadas. |

**Ejemplos:**

* La siguiente celda cuenta los nulos de cada columna. Sólo `descuento` tiene nulos.

In [ ]:
ventas.null_count()

* La siguiente celda compara el promedio del descuento ignorando los nulos (comportamiento por omisión) con el que resulta de tratarlos como `0`.

In [ ]:
ventas.select(
    pl.col('descuento').mean().alias('ignorando_nulos'),
    pl.col('descuento').fill_null(0).mean().alias('nulos_como_cero'),
)

* La siguiente celda elimina los renglones sin descuento y muestra cuántos quedan.

In [ ]:
ventas.drop_nulls(subset='descuento').height

* La siguiente celda muestra la diferencia entre `NaN` y nulo: sólo el valor `None` cuenta como nulo.

In [ ]:
pl.DataFrame({'x': [1.0, None, float('nan')]}).with_columns(
    es_nulo=pl.col('x').is_null(),
    es_nan=pl.col('x').is_nan(),
)

## Agrupación y agregación.

```
df.group_by(<claves>, maintain_order=<booleano>).agg(<expresiones>)
```

Donde:

* `<claves>` son una o varias columnas (o expresiones) que definen los grupos.
* `maintain_order=True` conserva los grupos en el orden en que aparecen por primera vez. Por omisión el orden no está garantizado, porque los grupos se calculan en paralelo.
* `<expresiones>` son las agregaciones; cada una produce una columna del resultado.

| Expresión | Resultado por grupo |
| :--- | :--- |
| `pl.len()` | Número de renglones. |
| `pl.col(...).sum()`, `.mean()`, `.median()`, `.min()`, `.max()`, `.std()` | Estadísticas de la columna. |
| `pl.col(...).n_unique()` | Número de valores distintos. |
| `pl.col(...).first()`, `.last()` | Primer o último valor. |

El objeto agrupado también tiene atajos como `len()`, `sum()` o `mean()`, que aplican la agregación a todas las columnas que no son claves.

[Documentación de `group_by()`](https://docs.pola.rs/api/python/stable/reference/dataframe/group_by.html)

**Ejemplos:**

* La siguiente celda calcula, por categoría, el número de ventas, las piezas vendidas, el importe neto total y el número de productos distintos; después ordena por importe.

In [ ]:
(
    ventas
    .group_by('categoria')
    .agg(
        pl.len().alias('ventas'),
        pl.col('cantidad').sum().alias('piezas'),
        pl.col('importe_neto').sum().round(2).alias('importe_neto'),
        pl.col('producto').n_unique().alias('productos'),
    )
    .sort('importe_neto', descending=True)
)

* La siguiente celda agrupa por dos claves, categoría y canal, y calcula el ticket promedio.

In [ ]:
(
    ventas
    .group_by('categoria', 'canal')
    .agg(pl.col('importe_neto').mean().round(2).alias('ticket_promedio'))
    .sort('categoria', 'canal')
)

* La siguiente celda usa como clave una expresión: el año de la venta, que se extrae de `fecha_hora` con `dt.year()`.

In [ ]:
ventas.group_by(pl.col('fecha_hora').dt.year().alias('anio')).agg(
    pl.col('importe_neto').sum().round(2)
).sort('anio')

* Un objeto agrupado también se puede recorrer con un ciclo `for`. Cada elemento es una tupla con la clave y el *dataframe* del grupo. La clave es a su vez una tupla, con un valor por cada columna de agrupación. La siguiente celda recorre los grupos por canal y muestra cuántas ventas tiene cada uno.

In [ ]:
for (canal,), grupo in ventas.group_by('canal', maintain_order=True):
    print(f"{canal:10s} {grupo.height:6,} ventas")

## *Polars*, *pandas* y *Arrow*.

En un proyecto real, *Polars* casi nunca trabaja solo: *Seaborn*, *statsmodels* o *scikit-learn* esperan un *dataframe* de *pandas*; otros sistemas entregan o reciben datos en formato de *Arrow*; y algunos cálculos numéricos se hacen con *NumPy*. Esta sección presenta las funciones que conectan a *Polars* con esas bibliotecas, explica cuándo una conversión copia los datos y cuándo no, y termina con un ejemplo que usa las tres bibliotecas de datos tabulares en un mismo proceso.

| Función o método | Dirección |
| :--- | :--- |
| [`pl.from_pandas()`](https://docs.pola.rs/api/python/stable/reference/api/polars.from_pandas.html) | *pandas* → *Polars* |
| [`df.to_pandas()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.to_pandas.html) | *Polars* → *pandas* |
| [`pl.from_arrow()`](https://docs.pola.rs/api/python/stable/reference/api/polars.from_arrow.html) | *PyArrow* → *Polars* |
| [`df.to_arrow()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.to_arrow.html) | *Polars* → *PyArrow* |
| [`df.to_numpy()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.to_numpy.html) | *Polars* → *NumPy* |
| [`df.to_dicts()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.to_dicts.html) | *Polars* → lista de diccionarios de *Python*, uno por renglón |

### De *pandas* a *Polars*: `pl.from_pandas()`.

```
pl.from_pandas(<df>, schema_overrides=<tipos>, rechunk=<booleano>, nan_to_null=<booleano>, include_index=<booleano>)
```

Donde:

* `<df>` es un *dataframe* o una serie de *pandas*.
* `schema_overrides` es opcional: un diccionario `{columna: tipo}` que fija el tipo de *Polars* de algunas columnas; las demás se infieren.
* `rechunk` indica si cada columna del resultado se reúne en un solo bloque contiguo de memoria. Por omisión es `True`, lo que agiliza las operaciones posteriores.
* `nan_to_null` indica si los `NaN` de las columnas de punto flotante se convierten en nulos de *Polars*. Por omisión es `True`, porque *pandas* usa `NaN` para representar los datos faltantes.
* `include_index` indica si el índice de *pandas* se agrega como una columna. Por omisión es `False`: como *Polars* no tiene índice de renglones, el índice se descarta.

### De *Polars* a *pandas*: `to_pandas()`.

```
<df>.to_pandas(use_pyarrow_extension_array=<booleano>)
```

Donde:

* `<df>` es un *dataframe* de *Polars*.
* `use_pyarrow_extension_array` indica qué tipos usarán las columnas del resultado. Con `False` (por omisión), los números usan los tipos de *NumPy* y el texto el tipo `str` de *pandas* 3; los datos se copian. Con `True`, todas las columnas usan tipos de *Arrow* (`pd.ArrowDtype`, visto en el *notebook* 34), los nulos se conservan como nulos y, en general, los datos no se copian.

**Ejemplos:**

* La siguiente celda crea un *dataframe* de *pandas* con un índice con nombre y un dato faltante, y lo convierte a *Polars* de dos maneras: con las opciones por omisión y conservando el índice y los `NaN`.

In [ ]:
import numpy as np
import pandas as pd

existencias_pd = pd.DataFrame(
    {'piezas': [12.0, np.nan, 7.0]},
    index=pd.Index(['A-01', 'A-02', 'A-03'], name='clave'),
)

print(pl.from_pandas(existencias_pd))
print(pl.from_pandas(existencias_pd, include_index=True, nan_to_null=False))

* En el primer resultado, el índice desapareció y el `NaN` se convirtió en nulo. En el segundo, `clave` es una columna y el `NaN` se conservó. La siguiente celda muestra por qué la diferencia importa: `null_count()` sólo cuenta los nulos, no los `NaN`.

In [ ]:
print(pl.from_pandas(existencias_pd)['piezas'].null_count())
print(pl.from_pandas(existencias_pd, nan_to_null=False)['piezas'].null_count())

* La siguiente celda convierte un resumen de *Polars* a *pandas* con las dos opciones de `use_pyarrow_extension_array` y compara los tipos de las columnas.

In [ ]:
resumen = ventas.group_by('categoria').agg(
    pl.len().alias('ventas'), pl.col('importe_neto').sum().round(2)
).sort('categoria')

print(resumen.to_pandas().dtypes, end='\n\n')
print(resumen.to_pandas(use_pyarrow_extension_array=True).dtypes)

### De *Arrow* a *Polars* y de vuelta: `pl.from_arrow()` y `to_arrow()`.

```
pl.from_arrow(<datos>, schema=<esquema>, schema_overrides=<tipos>, rechunk=<booleano>)
<df>.to_arrow(compat_level=<nivel>)
```

Donde:

* `<datos>` es una *table* o un *array* de *PyArrow*. Una *table* produce un *dataframe* y un *array* produce una serie.
* `schema` es opcional: la lista de nombres (o un diccionario `{nombre: tipo}`) que reemplaza a los nombres y tipos de las columnas de la *table*.
* `schema_overrides` es opcional: un diccionario `{columna: tipo}` que cambia el tipo sólo de algunas columnas.
* `rechunk` indica si cada columna se reúne en un solo bloque contiguo de memoria. Por omisión es `False`, de modo que, en general, los datos no se copian.
* `<df>` es un *dataframe* de *Polars*.
* `compat_level` es opcional y controla qué tipos de *Arrow* se usan al exportar. Por omisión, el texto se exporta como `large_string`, que entienden casi todas las herramientas; `pl.CompatLevel.newest()` usa los tipos más recientes (por ejemplo, `string_view`) y `pl.CompatLevel.oldest()` los más compatibles.

**Ejemplos:**

* La siguiente celda convierte el resumen a una *table* de *PyArrow* y muestra su *schema* con los dos niveles de compatibilidad.

In [ ]:
import pyarrow as pa

resumen_arrow = resumen.to_arrow()
print(resumen_arrow.schema, end='\n\n')
print(resumen.to_arrow(compat_level=pl.CompatLevel.newest()).schema)

* La siguiente celda regresa la *table* a *Polars*: primero con sus nombres y tipos, y después cambiando el tipo de la columna `ventas` a un entero de 32 bits con `schema_overrides`.

In [ ]:
print(pl.from_arrow(resumen_arrow).equals(resumen))
pl.from_arrow(resumen_arrow, schema_overrides={'ventas': pl.Int32}).schema

### De *Polars* a *NumPy*: `to_numpy()`.

```
<df>.to_numpy(order=<orden>, writable=<booleano>, allow_copy=<booleano>, structured=<booleano>, masked=<booleano>)
```

Donde:

* `<df>` es un *dataframe* de *Polars*. Las series tienen un método `to_numpy()` con algunos de estos mismos argumentos.
* `order` es el orden en memoria del arreglo de dos dimensiones: `'fortran'` (por omisión, columna por columna, que suele ser más rápido) o `'c'` (renglón por renglón).
* `writable` indica si el arreglo debe poder modificarse. Por omisión es `False`; con `True` se fuerza una copia cuando el arreglo comparte la memoria de *Polars*, que no se puede modificar.
* `allow_copy` indica si se permite copiar los datos. Por omisión es `True`; con `False` se desencadena una excepción si la conversión no puede hacerse sin copia.
* `structured` indica si se regresa un arreglo estructurado de *NumPy*, con un tipo de dato por columna, en lugar de un arreglo de dos dimensiones con un solo tipo. Por omisión es `False`.
* `masked` indica si se regresa un arreglo enmascarado (`numpy.ma`), en el que los nulos quedan ocultos por una máscara. Por omisión es `False`; es una función experimental de *Polars*.

**Ejemplos:**

* La siguiente celda convierte dos columnas numéricas a un arreglo de *NumPy*. Como los tipos son distintos (entero y flotante), el arreglo usa el tipo común, `float64`.

In [ ]:
arreglo = ventas.select('cantidad', 'precio_unitario').head(3).to_numpy()
print(arreglo.dtype)
arreglo

* Al leer un archivo en paralelo, una columna puede quedar repartida en varios bloques de memoria, y reunirlos en un arreglo de *NumPy* exige copiar. `rechunk()` reúne la columna en un solo bloque contiguo (esta operación sí copia). La siguiente celda convierte la columna reunida con `allow_copy=False`: la conversión funciona sin copiar los datos (*zero-copy*), y por eso el arreglo resultante es de sólo lectura.

In [ ]:
cantidades = ventas['cantidad'].rechunk().to_numpy(allow_copy=False)
print(cantidades[:5], cantidades.flags.writeable)

* La columna `descuento` tiene nulos, y *NumPy* no tiene un valor nulo para los números: los nulos deben convertirse en `NaN`, lo que exige copiar. La siguiente celda lo intenta con `allow_copy=False`, por lo que se desencadenará una excepción de tipo `RuntimeError`.

In [ ]:
ventas['descuento'].to_numpy(allow_copy=False)

### ¿Cuándo se copian los datos?

Como se vio en el *notebook* 34, sólo hay *zero-copy* cuando el formato de origen y el de destino coinciden. Para los datos de este curso:

| Conversión | ¿Copia los datos? |
| :--- | :--- |
| *Polars* → *PyArrow* (`to_arrow()`) | Casi nunca: ambos usan el formato de *Arrow*. La excepción son las columnas `Categorical`. |
| *PyArrow* → *Polars* (`pl.from_arrow()`) | Casi nunca, salvo con `rechunk=True` o cuando hay que convertir un tipo que *Polars* no maneja. |
| *Polars* → *pandas* (`to_pandas()`) | Sí, salvo con `use_pyarrow_extension_array=True`. |
| *pandas* → *Polars* (`pl.from_pandas()`) | Por lo general sí, porque *pandas* guarda casi todas las columnas con *NumPy*. |
| *Polars* → *NumPy* (`to_numpy()`) | No, para una columna numérica sin nulos; sí, cuando hay nulos, texto o varias columnas que deben reunirse en un solo arreglo. |

### Nulo y `NaN`.

*Polars* distingue dos situaciones que *pandas* suele representar con el mismo `NaN`:

* **Nulo** (`null`): el dato **falta**. Existe para cualquier tipo de columna; `is_null()`, `fill_null()` y `null_count()` lo detectan y lo tratan.
* **`NaN`** (*not a number*): el dato existe, pero es el resultado de una operación numérica no definida, como `0 / 0`. Sólo existe en columnas de punto flotante; lo detectan `is_nan()` y `fill_nan()`.

Por eso `pl.from_pandas()` convierte por omisión los `NaN` en nulos (`nan_to_null=True`), y `to_numpy()` convierte los nulos en `NaN`.

* La siguiente celda muestra ambos casos en una misma columna: el nulo de la entrada se conserva, y la división `0 / 0` produce un `NaN`.

In [ ]:
pl.DataFrame({'valor': [0.0, 2.0, None]}).with_columns(
    cociente=pl.col('valor') / pl.col('valor'),
).with_columns(
    es_nulo=pl.col('cociente').is_null(),
    es_nan=pl.col('cociente').is_nan(),
)

### Ejemplo completo: *Polars*, *pandas* y *PyArrow* en un mismo proceso.

El siguiente ejemplo reparte el trabajo según el punto fuerte de cada biblioteca:

1. *Polars* limpia y agrega las ventas: calcula el importe neto mensual por canal.
2. *pandas* recibe el resultado con `to_pandas()` para graficarlo con *Seaborn* (*notebook* 32).
3. El resultado se guarda en *Parquet* y *PyArrow* lo lee como una *table*, por ejemplo para entregarlo a otro sistema.

* La siguiente celda hace la limpieza y la agregación con *Polars*: descarta las ventas de la tienda que no existe en el catálogo (`tienda_id` 99, uno de los defectos intencionales descritos en el *notebook* 34), lleva cada fecha al primer día de su mes con `dt.truncate('1mo')`, conserva sólo la fecha con `dt.date()` (el espacio de nombres `.dt` se estudia a fondo en el *notebook* 36) y suma el importe neto por mes y canal.

In [ ]:
mensual = (
    ventas
    .filter(pl.col('tienda_id') != 99)
    .group_by(pl.col('fecha_hora').dt.truncate('1mo').dt.date().alias('mes'), 'canal')
    .agg(pl.col('importe_neto').sum().round(2).alias('importe'))
    .sort('mes', 'canal')
)
mensual.head(4)

* La siguiente celda convierte el resultado a *pandas* y lo grafica con `sns.lineplot()`: una línea por canal.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

mensual_pd = mensual.to_pandas()
ax = sns.lineplot(data=mensual_pd, x='mes', y='importe', hue='canal', marker='o')
ax.set(xlabel='mes', ylabel='importe neto')
plt.show()

* La siguiente celda guarda el resultado en *Parquet* con *Polars*, lo lee con `pyarrow.parquet.read_table()` y comprueba que, al regresar a *Polars*, los datos son los mismos.

In [ ]:
import pyarrow.parquet as pq

ruta_mensual = os.path.join(dir_tmp, 'ventas_mensuales_polars.parquet')
mensual.write_parquet(ruta_mensual)

mensual_arrow = pq.read_table(ruta_mensual)
print(mensual_arrow.schema, end='\n\n')
pl.from_arrow(mensual_arrow).equals(mensual)

### ¿Cuándo usar cada biblioteca?

| Biblioteca | Conviene para | En la ruta de cursos |
| :--- | :--- | :--- |
| *pandas* | El ecosistema de estadística, aprendizaje automático y gráficos (*Seaborn*, *statsmodels*, *scikit-learn*), que espera sus *dataframes*. | Es la base de los cursos posteriores de análisis estadístico, como py401. |
| *Polars* | Transformaciones y agregaciones sobre datos medianos o grandes, y el modo diferido (*notebook* 38). | Se usa para preparar los datos antes de entregarlos a otras herramientas. |
| *PyArrow* | El formato de los datos en memoria y en disco (*Parquet*, *Feather*) y el intercambio con otros sistemas. | Es el formato común de *pandas* 3, *Polars* y otras herramientas de datos. |

Una práctica común es hacer la carga y las transformaciones con *Polars* y convertir a *pandas* sólo al final, cuando una biblioteca lo requiere.

* La siguiente celda convierte los dos primeros renglones del catálogo de productos a una lista de diccionarios con `to_dicts()`, útil para entregar pocos datos a código de *Python* puro.

In [ ]:
productos.head(2).to_dicts()

## Resumen.

| Concepto | Descripción |
| :--- | :--- |
| ***Polars*** | Biblioteca de *dataframes* en *Rust*, columnar (*Arrow*), multihilo y sin índice de renglones. |
| ***Dataframe* y serie** | Tabla de columnas con tipo fijo; los renglones se obtienen por posición. |
| **Lectura y escritura** | `read_*` y `write_*` para *CSV*, *Parquet*, *Feather*, *JSON*, *Excel* y bases de datos. |
| **Contextos** | `select()`, `with_columns()`, `filter()` y `group_by().agg()` evalúan expresiones y regresan un *dataframe* nuevo. |
| **Selección de columnas** | Por nombre, expresión regular (`^...$`), tipo o selectores (`cs`). |
| **Nulos** | `null` en cualquier tipo, distinto de `NaN`; `fill_null()`, `drop_nulls()`, `null_count()`. |
| **Agrupación** | `group_by(...).agg(...)` con una o varias claves; el orden de los grupos no está garantizado. |
| ***Polars*, *pandas* y *Arrow*** | `pl.from_pandas()`, `to_pandas()`, `pl.from_arrow()`, `to_arrow()` y `to_numpy()`; con *Arrow* casi nunca hay copia, con *pandas* y *NumPy* casi siempre. |
| **Nulo y `NaN`** | Nulo: falta el dato; `NaN`: resultado numérico no definido. `from_pandas()` convierte los `NaN` en nulos por omisión. |

El *notebook* 36 estudia las expresiones a fondo, el 37 la combinación y el remodelado de tablas, y el 38 el modo diferido y el trabajo con datos más grandes que la memoria.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>